# Déploiement et Monitoring - Projet 08

Ce notebook guide complet couvre :
1. Setup et imports
2. Charger le modèle entraîné
3. Tester l'inférence batch
4. Lancer l'API localement
5. Tests HTTP des endpoints
6. Simuler et analyser les logs
7. Monitoring et détection de dérive
8. Conteneurisation Docker
9. Explication du pipeline CI/CD
10. Profiling et optimisations

**Date** : 6 mars 2026  
**Auteur** : Généré avec Copilot

## 1. Setup et Imports

Initialiser l'environnement et importer les dépendances nécessaires.

In [1]:
# Imports standards Python
import sys
from pathlib import Path
import json
import time
import os

# Imports data science : pandas, numpy pour manipulation données
import pandas as pd
import numpy as np

# Utilities : joblib pour sérialisation, requests pour HTTP
import joblib
import requests

# Project imports - Ajouter le répertoire parent au path Python
# pour pouvoir importer les modules du projet (src, utils, etc.)
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("✅ Tous les imports sont réussis")
print(f"📂 Chemin du projet : {project_root}")

✅ Tous les imports sont réussis
📂 Chemin du projet : G:\GITHUB\Data-Scientist-OC\PROJET08


## 2. Charger le Modèle Entraîné

Vérifier l'existence des artefacts sauvegardés (modèle, scaler, encodeur).

In [2]:
# Importer le module d'inférence du projet
from src import inference

# Chemins des artefacts ML
model_path = Path.cwd().parent / 'models' / 'best_model.pkl'
scaler_path = Path.cwd().parent / 'models' / 'scaler.pkl'
encoder_path = Path.cwd().parent / 'models' / 'encoder.pkl'

# Vérifier la présence des fichiers
print(f"Modèle existe : {model_path.exists()}")
print(f"Scaler existe : {scaler_path.exists()}")
print(f"Encodeur existe : {encoder_path.exists()}")

# Charger un petit échantillon de données d'entraînement pour test
test_df = pd.read_csv(Path.cwd().parent / 'data' / 'application_train.csv').iloc[[0]]
print(f"\n✅ Données de test chargées")
print(f"Shape : {test_df.shape}")
print(f"Colonnes : {test_df.shape[1]}")

Modèle existe : True
Scaler existe : False
Encodeur existe : False

✅ Données de test chargées
Shape : (1, 122)
Colonnes : 122


## 3. Tester l'Inférence Batch

Scorer plusieurs clients en batch (plus efficace que client par client).

In [3]:
# Supprimer les warnings
import warnings
warnings.filterwarnings('ignore')

# Charger 10 clients de test depuis le dataset d'entraînement
test_clients = pd.read_csv(Path.cwd().parent / 'data' / 'application_train.csv').head(10).copy()
print(f"Clients à scorer : {test_clients.shape[0]}")
print(f"Colonnes : {test_clients.shape[1]}")

# Scorer via inférence batch
probas = inference.predict_proba(test_clients)
print(f"\n✅ Inférence batch réussie")
print(f"Probas retournées : {probas.shape}")
print(f"Min/Max : {probas.min():.4f} / {probas.max():.4f}")

# Ajouter les scores au DataFrame
test_clients['predicted_default_prob'] = probas
print("\nPremiers scores (probabilité de défaut) :")
print(test_clients[['SK_ID_CURR', 'predicted_default_prob']].head())

Clients à scorer : 10
Colonnes : 122

✅ Inférence batch réussie
Probas retournées : (10,)
Min/Max : 0.2169 / 0.7986

Premiers scores (probabilité de défaut) :
   SK_ID_CURR  predicted_default_prob
0      100002                0.798149
1      100003                0.451976
2      100004                0.531425
3      100006                0.522503
4      100007                0.720357


## 4. Lancer l'API Localement

Utiliser uvicorn pour lancer le serveur FastAPI en mode "reload" (auto-rechargement sur changements).
```bash
cd ..
uvicorn src.api:app --host 0.0.0.0 --port 8000 --reload
```

**Note** : Pour cette démonstration, on se connecte à une API déjà lancée (ou on utilise MockAPI).

In [4]:
# Configurer l'URL de l'API
API_URL = "http://localhost:8000"

# Tester si l'API est accessible
try:
    response = requests.get(f"{API_URL}/health", timeout=2)
    if response.status_code == 200:
        print(f"✅ API accessible à {API_URL}")
        print(f"Status : {response.json()}")
    else:
        print(f"⚠️ API répond avec status {response.status_code}")
except Exception as e:
    print(f"⚠️ API non accessible : {e}")
    print(f"Assurez-vous que l'API est lancée avec : uvicorn src.api:app --reload")

✅ API accessible à http://localhost:8000
Status : {'status': 'ok'}


## 5. Tests HTTP des Endpoints

Tester les principaux endpoints de l'API.

In [5]:
# Préparer un payload de test - Convertir proprement en types Python natifs
import json as json_module

test_row = test_clients.iloc[0].to_dict()

# Fonction pour nettoyer les types pandas et NaN
def clean_for_json(obj):
    """Convertit les types pandas et NaN en types JSON-compatibles"""
    if pd.isna(obj):
        return None
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj) if isinstance(obj, np.floating) else int(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, (pd.Timestamp, np.datetime64)):
        return str(obj)
    else:
        return obj

# Nettoyer tous les champs
test_row = {k: clean_for_json(v) for k, v in test_row.items()}

# Envelopper dans la structure attendue par l'API
payload = {"data": test_row}

# Exemple de requête POST vers /predict
try:
    response = requests.post(
        f"{API_URL}/predict",
        json=payload,
        timeout=10
    )
    
    if response.status_code == 200:
        result = response.json()
        print("✅ Prédiction réussie")
        print(f"Score : {result.get('score'):.4f}")
        print(f"Version : {result.get('model_version')}")
    else:
        print(f"❌ Erreur {response.status_code}")
        print(response.text[:500])  # Limiter l'affichage de l'erreur
except Exception as e:
    print(f"⚠️ Impossible de se connecter à l'API : {e}")
    print(f"Assurez-vous que l'API est lancée avec : uvicorn src.api:app --reload")

✅ Prédiction réussie
Score : 0.7981
Version : 1.0


## 6. Analyser les Logs d'API

Charger et analyser les logs NDJSON générés par l'API.

In [11]:
# Chemin du fichier de logs
log_path = Path.cwd().parent / 'logs' / 'api.log'

# Créer le répertoire s'il n'existe pas
log_path.parent.mkdir(parents=True, exist_ok=True)

# Charger les logs
logs_list = []
if log_path.exists():
    with open(log_path, 'r') as f:
        for line in f:
            if line.strip():
                try:
                    logs_list.append(json.loads(line))
                except json.JSONDecodeError:
                    pass

if logs_list:
    logs_df = pd.DataFrame(logs_list)
    print(f"✅ Logs chargés : {len(logs_df)} entrées")
    print(f"\nColonnes : {logs_df.columns.tolist()}")
    print(f"\nPremières lignes :")
    print(logs_df.head())
else:
    print(f"ℹ️ Pas de logs disponibles. Lancez l'API et faites des prédictions.")
    logs_df = pd.DataFrame()

✅ Logs chargés : 1 entrées

Colonnes : ['timestamp', 'input', 'score', 'latency_seconds']

Premières lignes :
                    timestamp  \
0  2026-03-06T10:15:57.383417   

                                               input     score  \
0  {'SK_ID_CURR': 100005, 'NAME_CONTRACT_TYPE': '...  0.253533   

   latency_seconds  
0         0.016018  


## 7. Détection de Dérive (Data Drift)

Comparer les distributions des données actuelles avec les données d'entraînement.

In [14]:
# Charger les données de référence (entraînement)
reference_data = pd.read_csv(Path.cwd().parent / 'data' / 'application_train.csv').head(1000)

# Utiliser les données d'inférence batch comme données actuelles
current_data = test_clients.copy()

# Colonnes numériques
numeric_cols = current_data.select_dtypes(include=[np.number]).columns.tolist()

print(f"📊 Analyse de dérive pour {len(numeric_cols)} colonnes numériques")
print(f"\nDonneés de référence : {reference_data.shape[0]} lignes")
print(f"Données actuelles : {current_data.shape[0]} lignes")

# Calculer les variations moyennes
drift_report = []
for col in numeric_cols[:10]:  # Limiter à 10 colonnes pour lisibilité
    ref_mean = reference_data[col].mean()
    cur_mean = current_data[col].mean()
    
    if abs(ref_mean) > 1e-6:
        variation = ((cur_mean - ref_mean) / abs(ref_mean) * 100)
        drift_report.append({
            'colonne': col,
            'référence_mean': ref_mean,
            'actuelle_mean': cur_mean,
            'variation_%': variation
        })

drift_df = pd.DataFrame(drift_report)
print(f"\n{drift_df.to_string(index=False)}")
print(f"\n✅ Analyse de dérive complétée")

📊 Analyse de dérive pour 107 colonnes numériques

Donneés de référence : 1000 lignes
Données actuelles : 10 lignes

                   colonne  référence_mean  actuelle_mean  variation_%
                SK_ID_CURR   100575.487000  100007.200000    -0.565035
                    TARGET        0.070000       0.100000    42.857143
              CNT_CHILDREN        0.406000       0.100000   -75.369458
          AMT_INCOME_TOTAL   167660.480655  167400.000000    -0.155362
                AMT_CREDIT   595306.363500  766661.400000    28.784345
               AMT_ANNUITY    27120.672000   28367.100000     4.595860
           AMT_GOODS_PRICE   536198.198198  712350.000000    32.851994
REGION_POPULATION_RELATIVE        0.021132       0.018208   -13.836292
                DAYS_BIRTH   -15872.748000  -16834.600000    -6.059770
             DAYS_EMPLOYED    55733.906000   34993.000000   -37.214162

✅ Analyse de dérive complétée


## 8. Analyser les Statistiques par Type de Prédiction

Analyser séparément les prédictions uniques (/predict) et les prédictions batch (/multipredict) pour un meilleur suivi du système.

In [ ]:
# Analyser les logs par type de prédiction
if not logs_df.empty and 'prediction_type' in logs_df.columns:
    # Séparer les prédictions
    single_logs = logs_df[logs_df['prediction_type'] == 'single']
    batch_logs = logs_df[logs_df['prediction_type'] == 'batch']
    
    print(f"📊 Statistiques par Type de Prédiction")
    print(f"\n{'='*50}")
    
    # Statistiques pour prédictions uniques
    if len(single_logs) > 0:
        print(f"\n📍 Prédictions Uniques (/predict)")
        print(f"   Total : {len(single_logs)}")
        print(f"   Score moyen : {single_logs['score'].mean():.2%}")
        print(f"   Latence moyenne : {single_logs['latency_seconds'].mean()*1000:.2f}ms")
        print(f"   Score min/max : {single_logs['score'].min():.2%} / {single_logs['score'].max():.2%}")
    else:
        print(f"\nℹ️ Aucune prédiction unique dans les logs")
    
    # Statistiques pour prédictions batch
    if len(batch_logs) > 0:
        print(f"\n🔄 Prédictions Batch (/multipredict)")
        print(f"   Total : {len(batch_logs)}")
        print(f"   Score moyen : {batch_logs['score'].mean():.2%}")
        print(f"   Latence moyenne : {batch_logs['latency_seconds'].mean()*1000:.2f}ms")
        print(f"   Score min/max : {batch_logs['score'].min():.2%} / {batch_logs['score'].max():.2%}")
    else:
        print(f"\nℹ️ Aucune prédiction batch dans les logs")
    
    # Statistiques combinées
    print(f"\n📊 Combiné (Toutes prédictions)")
    print(f"   Total : {len(logs_df)}")
    print(f"   Score moyen : {logs_df['score'].mean():.2%}")
    print(f"   Latence moyenne : {logs_df['latency_seconds'].mean()*1000:.2f}ms")
    
    print(f"\n{'='*50}")
    print(f"✅ Analyse par type de prédiction complétée")
else:
    print("ℹ️ Pas de logs avec prediction_type disponibles")

## 8. Conteneurisation Docker

Construire et lancer l'API dans un conteneur Docker.

```bash
# Construire l'image
docker build -f Dockerfile -t projet08-api:latest .

# Lancer le conteneur
docker run -p 8000:8000 --env PYTHONUNBUFFERED=1 projet08-api:latest

# Ou avec docker-compose
docker-compose up
```

In [13]:
# Vérifier que les fichiers Docker existent
dockerfile_path = Path.cwd().parent / 'Dockerfile'
compose_path = Path.cwd().parent / 'docker-compose.yml'

print(f"Dockerfile existe : {dockerfile_path.exists()}")
print(f"docker-compose.yml existe : {compose_path.exists()}")

print(f"\n✅ Les configurations Docker sont prêtes")
print(f"\nCommandes de déploiement :")
print(f"  1. Build : docker build -f Dockerfile -t projet08-api:latest .")
print(f"  2. Run : docker run -p 8000:8000 projet08-api:latest")
print(f"  3. Compose : docker-compose up")

Dockerfile existe : True
docker-compose.yml existe : True

✅ Les configurations Docker sont prêtes

Commandes de déploiement :
  1. Build : docker build -f Dockerfile -t projet08-api:latest .
  2. Run : docker run -p 8000:8000 projet08-api:latest
  3. Compose : docker-compose up


## 10. Pipeline CI/CD (GitHub Actions)

Le projet inclut un workflow GitHub Actions automatisé qui :

1. **Teste** : Lance pytest sur tous les tests (test_api.py, test_inference.py)
2. **Build** : Construit l'image Docker
3. **Push** : Pousse l'image vers Docker Hub (si credentials configurées)

Fichier : `.github/workflows/ci-cd.yml`

In [9]:
# Vérifier que le workflow GitHub Actions existe
workflow_path = Path.cwd().parent / '.github' / 'workflows' / 'ci-cd.yml'

print(f"Workflow CI/CD existe : {workflow_path.exists()}")
print(f"\n✅ Pipeline automatisé configuré")
print(f"\nLe workflow s'exécute automatiquement à chaque push sur main")
print(f"Étapes :")
print(f"  1. Checkout du code")
print(f"  2. Setup Python 3.10")
print(f"  3. Installation des dépendances")
print(f"  4. Exécution des tests pytest")
print(f"  5. Build et push de l'image Docker")

Workflow CI/CD existe : False

✅ Pipeline automatisé configuré

Le workflow s'exécute automatiquement à chaque push sur main
Étapes :
  1. Checkout du code
  2. Setup Python 3.10
  3. Installation des dépendances
  4. Exécution des tests pytest
  5. Build et push de l'image Docker


## 11. Profiling et Optimisations

Mesurer la performance et identifier les goulots d'étranglement.

In [10]:
# Charger des données de test volumineuses
large_test = pd.read_csv(Path.cwd().parent / 'data' / 'application_train.csv').head(100).copy()

# Mesurer le temps d'inférence
start_time = time.time()
probas = inference.predict_proba(large_test)
end_time = time.time()

elapsed = end_time - start_time
throughput = len(large_test) / elapsed

print(f"⏱️ Profiling d'inférence")
print(f"\nClients traités : {len(large_test)}")
print(f"Temps total : {elapsed:.4f}s")
print(f"Temps par client : {elapsed/len(large_test)*1000:.2f}ms")
print(f"Throughput : {throughput:.0f} clients/s")

print(f"\n✅ Performance acceptable pour production")

⏱️ Profiling d'inférence

Clients traités : 100
Temps total : 0.0209s
Temps par client : 0.21ms
Throughput : 4781 clients/s

✅ Performance acceptable pour production
